# Expense Baseline

This notebook builds a partial YTD expense baseline from the cleaned Quicken transaction export.

Quicken remains authoritative for non-Amazon expenses. Transfer rows, mortgage, State Farm, and Amazon payees are excluded. Amazon monthly category summaries are added in their place.

Food is an approximate split: canonical Food expenses charged to Hiromi FECU Checking are assigned to Hiromi Food; all other Food expenses, including Amazon, are assigned to Bob Food.

Mortgage and State Farm are modeled separately in the scenario rather than inferred here.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from analysis.amazon.notebook import load_monthly_summaries
from analysis.expenses.baseline import build_expense_baseline, load_category_map

quicken_path = Path('../data/shared/quicken/Transaction Export.csv')
category_map_path = Path('../data/shared/quicken/expense_category_map.csv')
state_dir = Path('../state')

if not quicken_path.exists():
    raise FileNotFoundError(f'Quicken export not found at: {quicken_path.resolve()}')
if not category_map_path.exists():
    raise FileNotFoundError(
        f'Expense category map not found at: {category_map_path.resolve()}'
    )

category_merges, excluded_categories = load_category_map(category_map_path)
amazon_monthly_summaries = load_monthly_summaries(state_dir)
baseline = build_expense_baseline(
    quicken_path,
    amazon_monthly_summaries,
    category_merges=category_merges,
    excluded_categories=excluded_categories,
    default_category='Other',
    food_owner_account='Hiromi FECU Checking',
)

print(f'Included Quicken expense rows: {len(baseline.transactions):,}')
print(f'Excluded Quicken rows: {len(baseline.excluded):,}')
print(f'Amazon summary rows: {len(amazon_monthly_summaries):,}')
print('Categories defaulted to Other:', baseline.categories_defaulted_to_other)

## Categories Defaulted to Other

These categories are not explicitly canonicalized or excluded in the private category map. They currently collapse into `Other` and can be promoted later by adding a row to that map.

In [ ]:
pd.DataFrame({'category': baseline.categories_defaulted_to_other})

## Category Totals

In [ ]:

ct = baseline.category_totals.style.format({"amount_total": "${:,.2f}", "monthly": "${:,.2f}"})
with pd.option_context("display.max_rows", None):
    display(ct)

In [ ]:
# Divide by the number of *distinct months present in the export*, not a
# fixed span -- a month with no rows at all (a data gap) doesn't count
# toward the period, matching build_scheduled_baseline_rows's own
# definition (analysis.expenses.baseline), so this preview and the
# scheduled rows it will generate agree.
observed_months = baseline.monthly["summary_month_start"].nunique()
print(f"Months observed in this export: {observed_months}")

ct = baseline.category_totals.copy()
ct["monthly"] = ct["amount_total"] / observed_months
ct = ct.style.format({"amount_total": "${:,.2f}", "monthly": "${:,.2f}"})
with pd.option_context("display.max_rows", None):
    display(ct)

## Monthly Baseline

In [ ]:
monthly_baseline = (
    baseline.monthly.groupby(['summary_month_start', 'category'], as_index=False)['amount']
    .sum()
    .rename(columns={'amount': 'amount_total'})
    .sort_values(['summary_month_start', 'category'])
)
with pd.option_context("display.max_rows", None):
    display(monthly_baseline)

## Coverage and Exclusions

In [ ]:
excluded_summary = (
    baseline.excluded.assign(amount=lambda frame: frame['amount_cents'].abs() / 100)
    .groupby('exclude_reason', as_index=False)
    .agg(rows=('source_row', 'size'), amount=('amount', 'sum'))
    .sort_values('amount', ascending=False)
)
excluded_summary

In [ ]:
top_uncategorized = (
    baseline.transactions[baseline.transactions['category'].eq('Uncategorized')]
    .assign(amount=lambda frame: frame['amount_cents'].abs() / 100)
    .sort_values('amount', ascending=False)
    [['date', 'account', 'payee', 'amount', 'memo', 'source_row']]
    .head(25)
)
top_uncategorized

## Trend

In [ ]:
trend = monthly_baseline.pivot_table(
    index='summary_month_start',
    columns='category',
    values='amount_total',
    aggfunc='sum',
    fill_value=0.0,
)
top_categories = baseline.category_totals.head(8)['category'].tolist()
ax = trend[top_categories].plot(figsize=(11, 5), marker='o')
ax.set_title('Baseline Expense Categories')
ax.set_xlabel('Month')
ax.set_ylabel('Amount')
ax.legend(title='Category', loc='best')
plt.tight_layout()
plt.show()